In [1]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_color: str

d:\workspace\AI\langchain-academy\intro-2-langchain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


# Write to state

In [5]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [11]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5")
agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)


In [12]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [13]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='1ca9178b-799b-4a7d-9b0b-19c36f7f7d7c'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-04T18:48:09.0772074Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7649526000, 'load_duration': 6305506600, 'prompt_eval_count': 288, 'prompt_eval_duration': 116418600, 'eval_count': 114, 'eval_duration': 1091425800, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d59d2-d07d-7bb2-854b-c779356eedb8-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'afdfc347-4838-4b53-9ae1-317ca2b261fb', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 288, 'output_tokens': 114, 'total_tokens': 402}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id='

In [14]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='2db26530-8c3b-415d-8bf7-fb2fae013a5b'),
              AIMessage(content="Hello! I'm doing well, thank you for asking! How are you today? Is there anything I can help you with?", additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-04T18:49:04.6186424Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1061556800, 'load_duration': 127072000, 'prompt_eval_count': 289, 'prompt_eval_duration': 254173000, 'eval_count': 67, 'eval_duration': 652183700, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d59d3-c344-7bf2-89d9-5ae1a0f10832-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 289, 'output_tokens': 67, 'total_tokens': 356})]}


# Read State

In [15]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [16]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='53372e0e-1a9d-4152-a0aa-6c2d00bd49ee'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-04T18:53:08.7318791Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1439391500, 'load_duration': 118518900, 'prompt_eval_count': 337, 'prompt_eval_duration': 315000300, 'eval_count': 100, 'eval_duration': 958976400, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d59d7-7b5b-7c70-b83e-2a3eb2d28f39-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': '926900e4-1100-40d3-913f-4d7270bc69d3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 337, 'output_tokens': 100, 'total_tokens': 437}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id='2b

In [17]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='53372e0e-1a9d-4152-a0aa-6c2d00bd49ee'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-04T18:53:08.7318791Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1439391500, 'load_duration': 118518900, 'prompt_eval_count': 337, 'prompt_eval_duration': 315000300, 'eval_count': 100, 'eval_duration': 958976400, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d59d7-7b5b-7c70-b83e-2a3eb2d28f39-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': '926900e4-1100-40d3-913f-4d7270bc69d3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 337, 'output_tokens': 100, 'total_tokens': 437}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id='2b